# Colab'da 2D PINN Tam Eğitimi (GPU)

Bu defter, 2D radyal dam-break PINN'ini **GPU ile tam eğitir** (CPU'da ağır olan kısım).

## Kullanım (3 adım)
1. **GPU'yu aç:** Üst menü → `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Hücreleri **sırayla** çalıştır (Shift+Enter).
3. 3. hücre dosya isteyince bilgisayarından **`PINN-BAP-colab.zip`**'i seç.

> Tam 2D eğitim T4 GPU'da ~10-30 dk sürebilir. Daha hızlı deneme için 5. hücreden önce
> `config/param_ranges_2d.yaml` içinde `full.adam_iters`'i 8000-10000'e düşürebilirsin.

## 1) GPU kontrolü

In [ ]:
import torch
if torch.cuda.is_available():
    print('✓ GPU aktif:', torch.cuda.get_device_name(0))
else:
    print('⚠ GPU YOK! Runtime > Change runtime type > T4 GPU seç ve tekrar çalıştır.')

## 2) Bağımlılıklar (torch Colab'da hazır; sadece deepxde + pyyaml)

In [ ]:
!pip -q install deepxde pyyaml

## 3) Projeyi GitHub'dan çek
Zip yükleme yok — kod doğrudan repodan geliyor.
Bu hücre tekrar çalıştırılırsa son sürümü çeker (`git pull`).

In [ ]:
import os, subprocess

REPO = 'https://github.com/BurakkYuce/PINN-BAP.git'
if os.path.exists('/content/PINN-BAP'):
    subprocess.run(['git','-C','/content/PINN-BAP','pull','-q'])   # güncelle
else:
    subprocess.run(['git','clone','-q',REPO,'/content/PINN-BAP'])  # ilk çekiş
os.chdir('/content/PINN-BAP')
print('✓ Proje kökü:', os.getcwd())
print('  içerik:', sorted(os.listdir('.')))

## 4) Hızlı kontrol (opsiyonel) — smoke ile boru hattı çalışıyor mu?

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline_2d.py --smoke

## 5) TAM eğitim (GPU) — asıl iş
30k Adam + L-BFGS. Birkaç on dakika sürebilir; çıktı `results/2d/`'e yazılır.

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline_2d.py

## 6) Doğrulama — heatmap + radyal kesit + L2 hatası

In [ ]:
!python validation/compare_baseline_2d.py
from IPython.display import Image, display
display(Image('results/2d/h_heatmap.png'))
display(Image('results/2d/radial_cut.png'))

## 7) Sonuçları indir (kopmadan önce!)
Colab oturumu kapanınca dosyalar silinir; sonuçları zip'leyip indir.

In [ ]:
import shutil
shutil.make_archive('/content/PINN-BAP-2d-results', 'zip', 'results/2d')
from google.colab import files
files.download('/content/PINN-BAP-2d-results.zip')

---
### Not: 1D deneysel modeli de GPU'da eğitmek istersen
```
!python src/data/parse_vosoughi2021.py
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline.py --scenario experimental
!python validation/compare_experimental_pinn.py
```